In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ( classification_report, confusion_matrix, f1_score, precision_score, recall_score )

In [2]:
# 1. LOAD CLEANED DATA
df = pd.read_csv("fraud_clean.csv")
df.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,1
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0


In [9]:
# 2. TAKE A SMALL SAMPLE 
df = df.sample(frac=0.03, random_state=42)   # 3% of data
print("Sampled shape:", df.shape)

Sampled shape: (38138, 11)


In [13]:
# 3. ENCODE CATEGORICAL COLUMN
if "type" in df.columns:
    df = pd.get_dummies(df, columns=["type"], drop_first=True)
else:
    print("Column 'type' not found - data already encoded")

Column 'type' not found - data already encoded


In [14]:
# 4. TRAIN/TEST SPLIT
X = df.drop(columns=["isFraud"])
y = df["isFraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (30510, 10)
Test shape: (7628, 10)


In [ ]:
# 5. BASELINE MODEL
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=500,
    class_weight="balanced"
)

model.fit(X_train, y_train)
preds = model.predict(X_test)


print("\n--- BASELINE MODEL PERFORMANCE ---")
print(classification_report(y_test, preds))
print("Baseline F1 Score:", f1_score(y_test, preds))


--- BASELINE MODEL PERFORMANCE ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      7619
           1       1.00      0.89      0.94         9

    accuracy                           1.00      7628
   macro avg       1.00      0.94      0.97      7628
weighted avg       1.00      1.00      1.00      7628

Baseline F1 Score: 0.9411764705882353


In [27]:
# 2. Ultra-small parameter grid

param_grid = {
    "C": [0.1, 1, 10]
}


In [26]:
# 3. Safe GridSearch

log_reg = LogisticRegression(max_iter=500, class_weight="balanced")

grid = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid,
    scoring="f1",
    cv=2,
    verbose=1
)

grid.fit(X_train, y_train)

print("\nBest Hyperparameters:")
print(grid.best_params_)


Fitting 2 folds for each of 3 candidates, totalling 6 fits

Best Hyperparameters:
{'C': 10, 'l1_ratio': 0}


In [28]:
# 4. Retrain with best params

best_model = grid.best_estimator_
best_model.fit(X_train, y_train)

best_preds = best_model.predict(X_test)

print("\n--- OPTIMIZED MODEL PERFORMANCE ---")
print(classification_report(y_test, best_preds))
print("Optimized F1 Score:", f1_score(y_test, best_preds))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, best_preds))


--- OPTIMIZED MODEL PERFORMANCE ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      7619
           1       1.00      0.89      0.94         9

    accuracy                           1.00      7628
   macro avg       1.00      0.94      0.97      7628
weighted avg       1.00      1.00      1.00      7628

Optimized F1 Score: 0.9411764705882353

Confusion Matrix:
[[7619    0]
 [   1    8]]
